# Test Each Part of The Inference Pipeline

# Testing the Real World Cases with the BEST MODEL: BioBERT run_2!


In [ ]:
import os, json, torch, sys
from transformers import AutoModelForTokenClassification, AutoTokenizer


PARENT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PARENT_DIR not in sys.path:
    sys.path.insert(0, PARENT_DIR)

# Local imports 
from gcp_utils import download_from_gcs
from config import settings
from inference.v01.inference_utils import word_labels_to_spans, predict_word_level


In [ ]:

# ------- Load labels and test data - LOCALLY -------
with open("../v01/data/biobert_splits/test.jsonl", "r") as f:
    test_data = []
    for line in f:
        test_data.append(line)

with open("../v01/data/id2label.json", "r") as f:
    id2label = json.load(f)
with open("../v01/data/label2id.json", "r") as f:
    label2id = json.load(f)
# ------------------------------------------------------

# Convert id2label keys from strings to integers (JSON loads keys as strings)
if any(isinstance(k, str) for k in id2label.keys()):
    id2label = {int(k): v for k, v in id2label.items()}


In [ ]:
# CONFIG FOR LOADING FROM GCS 
VERSION = "v02"
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1" 
RUN_IDX = 2

GCS_MODEL_PATH = f"{VERSION}/runs/{MODEL_NAME}/run_{RUN_IDX}"
BUCKET_NAME = settings.BUCKET_NAME  # "ner_training_data_results"

RUN_ROOT = os.path.abspath(
    os.path.join("downloaded_models", *MODEL_NAME.split("/"), f"run_{RUN_IDX}")
)
# After the download/load cell runs, LOCAL_MODEL_DIR points at .../checkpoint-<step>/ inside RUN_ROOT.


In [ ]:
BUCKET_NAME

In [ ]:
GCS_MODEL_PATH, RUN_ROOT

In [ ]:
# Create a local directory path for the model
LOCAL_MODEL_DIR = f"./downloaded_models/{MODEL_NAME}/run_{RUN_IDX}"

# Check if model already exists locally
if os.path.exists(LOCAL_MODEL_DIR) and os.path.isfile(os.path.join(LOCAL_MODEL_DIR, "config.json")):
    print(f"✅ Model found locally at {LOCAL_MODEL_DIR}")
    print("Skipping download from GCS.")
else:
    # Download the model directory from GCS if not found locally
    print(f"📥 Model not found locally. Downloading from gs://{BUCKET_NAME}/{GCS_MODEL_PATH}...")
    downloaded_path = download_from_gcs(
        gcs_path=GCS_MODEL_PATH,
        local_path=LOCAL_MODEL_DIR,
        bucket_name=BUCKET_NAME
    )
    if downloaded_path:
        print(f"✅ Download complete. Model saved to {LOCAL_MODEL_DIR}")

# Load the model
print(f"📂 Loading model from {LOCAL_MODEL_DIR}...")
model = AutoModelForTokenClassification.from_pretrained(LOCAL_MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)

# Move to device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")
model.to(device)
model.eval()

# Print number of parameters and space in bytes
num_parameters = sum(p.numel() for p in model.parameters())
num_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
mega_bytes = num_bytes / 1_000_000
print(f"Number of parameters: {num_parameters}")
print(f"Model size (Mega bytes): {mega_bytes}")

# Token Level Prediction

In [ ]:
from inference.v01.inference_utils import predict_token_level
test_text = "Patient reports severe headache and nausea"
tokens, predictions = predict_token_level(test_text, model, tokenizer, device=device)
print(f"Tokens {len(tokens)}:\n\t{tokens}")
print(f"Predictions {len(predictions)}:\n\t{predictions}")

In [ ]:
# RUN ANOTHER EXAMPLE: 
sample = json.loads(test_data[1])
text = sample.get('text')
tokens = sample.get('tokens')
token_label_ids = sample.get('token_label_ids')
print(f"TEXT: {text}")
print(f"TOKENS from test data: {tokens}")
tks, predictions = predict_token_level(text, model, tokenizer, device=device)
print(f"Returned tokens: {tks}")
print(f"Predictions: {predictions[0]}")


# Word Level Prediction

In [ ]:
from inference.v01.inference_utils import predict_word_level

sample = json.loads(test_data[1])
text = sample.get('text')

tokens, token_level_labels, word_ids, words, word_level_labels, word_offsets = predict_word_level(
    text=text,
    model=model,
    tokenizer=tokenizer,
    id2label=id2label,
    device=device
)
print("tokens:", tokens)
print("token_level_labels:", token_level_labels)
print("word_ids:", word_ids)
print("words:", words)
print("word_level_labels:", word_level_labels)
print("word_offsets:", word_offsets)



In [ ]:
# RUN ANOTHER EXAMPLE
from inference.v01.inference_utils import predict_word_level

#sample = json.loads(test_data[1])
text = "The patient has cataplexy." #lymphatic system symptom and cataplexy."#sample.get('text')
samples = ["The patient has cataplexy.", "The patient has lymphatic system symptom.", "The patient has lymphatic system symptom and cataplexy.","The patient has lymphatic system symptom.", "The patient has cataplexy and lymphatic system symptom.", "Roberta does not have back pain but she has an inflamation in her wrist"]
for s in samples:
    print("="*20)
    print(f" TEXT: {s}")
    print("="*20)
    tokens,token_labels,word_ids, words, word_labels, word_offsets =  predict_word_level(text=s, model=model, tokenizer=tokenizer, id2label=id2label, device=device)
    print("tokens: ", tokens)
    print("token labels: ", token_labels)
    print("word_ids: ", word_ids)
    print("word: ", words)
    print("bio word labels: ", word_labels)
    print("word_offsets:", word_offsets)
    print()

# Test word_labels --> entity_spans

In [ ]:
examples = {
    "ex1" : [
        ['The', 'patient', 'has', 'lymphatic', 'system', 'symptom', '.'],
        ['O', 'O', 'O', 'B-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'O']
    ],
    "ex2" : [
        ['The', 'patient', 'has', 'cataplexy', '.'],
        ['O', 'O', 'O', 'B-SYMPTOM_POS', 'O']    
    ],
    "ex3" : [
        ['Roberta', 'does', 'not', 'have', 'back', 'pain', 'but', 'she', 'has', 'an', 'inflamation', 'in', 'her', 'wrist'],
        ['O', 'O', 'O', 'O', 'B-SYMPTOM_NEG', 'I-SYMPTOM_NEG', 'O', 'O', 'O', 'O', 'I-SYMPTOM_POS', 'I-SYMPTOM_NEG', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS']
    ]
}

from inference.v01.inference_utils import word_labels_to_spans

def get_text_and_offsets(words):
    """
    Construct a text string and offsets from a list of words.

    Returns:
        text: A single string joining the words with spaces.
        offsets: A list of (start, end) index tuples for each word in the joined text.
    """
    text = ""
    offsets = []
    cur = 0  # This tracks the current character index in the final text string.

    # Iterate through the list of words
    for i, word in enumerate(words):
        # Insert a space before each word except the first one
        if i > 0:
            text += " "
            cur += 1  # Account for the space in the character offset

        start = cur          # Mark the start of the current word
        text += word         # Add the current word to the reconstructed text
        end = cur + len(word)  # Compute the end index (exclusive) of the word

        offsets.append((start, end))  # Store the (start, end) tuple for each word

        cur = end  # Update the cursor to point to the end of the current word for the next iteration

    return text, offsets

ex1_words, ex1_labels = examples["ex1"]
ex1_text, ex1_offsets = get_text_and_offsets(ex1_words)

spans = word_labels_to_spans(text=ex1_text, word_offsets=ex1_offsets, word_labels=ex1_labels)
print("ex1 spans:", spans)

In [ ]:
# Import gold-standard cases
from real_world_cases import REAL_WORLD_CASES

for case in REAL_WORLD_CASES:
    text = case["text"]
    expected_entities = case.get("expected_entities", [])
    print(f"\n=== Case: {case['id']} ===")
    print(f"Text: {text}")

    tokens, token_labels, word_ids, words, word_labels, word_offsets = predict_word_level(
        text=text,
        model=model,
        tokenizer=tokenizer,
        id2label=id2label,
        device=device,
    )
    spans = word_labels_to_spans(text=text,
        word_offsets=word_offsets,
        word_labels=word_labels)
  
    spans = [span_data for span_data in spans if span_data['label'] != 'O']
    print("Tokens:", tokens)
    print("Token-level labels:", token_labels)
    print("Word-level label indices:", word_labels)
    print("Predicted spans (Excluding 'O'):", spans)
    print("Expected entities:", expected_entities)
